# 09 Stage 2 Logistic Regression — Health Outcome Prediction

**Owner:** PBC  
**Targets:** `target_unmet_fp` (and `target_anc_gap` when m14 is available)  
**Depends on:** `07_data_integration.ipynb`, `08_clustering.ipynb`

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.stage2_logistic import configure_logging, train_all_stage2_logistic
from src.models.stage2_xgboost import STAGE2_TARGETS, TARGET_DISPLAY, load_stage2_data

configure_logging()

In [2]:
# Ensure Stage 2 inputs exist
stage2_files = [
    PROJECT_ROOT / 'data/processed/stage2/X_stage2_preclustering.csv',
    PROJECT_ROOT / 'data/processed/stage2/y_stage2_targets.csv',
    PROJECT_ROOT / 'outputs/stage2_results/cluster_assignments.csv',
]
if not all(p.exists() for p in stage2_files):
    raise FileNotFoundError('Run scripts/run_stage2_data_prep.py or 08_clustering.ipynb first.')
print('Stage 2 inputs found.')

Stage 2 inputs found.


In [3]:
X_full, y = load_stage2_data()
print(f'Feature matrix (with cluster dummies): {X_full.shape}')
for col in y.columns:
    nn = y[col].notna().sum()
    pos = y.loc[y[col].notna(), col].sum() if nn else 0
    print(f'{col}: non-null N = {nn:,} | positive = {int(pos):,}')

2026-07-26 15:47:25,273 | INFO | src.models.stage2_xgboost | Loaded Stage 2 data: X=(724115, 64), targets=['target_unmet_fp']


Feature matrix (with cluster dummies): (724115, 64)
target_unmet_fp: non-null N = 466,859 | positive = 49,672


In [4]:
# Train separate LogisticRegression models per target (Section 6.2)
results = train_all_stage2_logistic()
metrics_df = results.get('metrics_df')
if metrics_df is not None:
    display_cols = ['Target', 'TrainSize', 'TestSize', 'ROC-AUC', 'F1-Score', 'CV_ROC-AUC', 'Barrier_Uplift']
    metrics_df[[c for c in display_cols if c in metrics_df.columns]]

2026-07-26 15:47:32,517 | INFO | src.models.stage2_xgboost | Loaded Stage 2 data: X=(724115, 64), targets=['target_unmet_fp']
2026-07-26 15:47:33,015 | INFO | src.models.stage2_xgboost | target_unmet_fp — analytic sample: 466859 rows (positive rate 0.1064)


--- Top predictors for target_unmet_fp ---
               Feature  Coefficient  OddsRatio
          v501_married     0.315614   1.371101
         v743f_missing     0.235262   1.265241
       v159_not at all     0.133069   1.142329
            v013_45-49     0.129843   1.138650
            v013_40-44     0.117484   1.124664
          v190_poorest     0.117055   1.124181
household_barrier_prob     0.102232   1.107641
      v717_not working     0.090095   1.094279
        v130_christian     0.082216   1.085690
            v013_25-29     0.077391   1.080465

=== Logistic Regression | target_unmet_fp barrier ===
  Model       : Logistic Regression
  Target      : target_unmet_fp
  Accuracy    : 0.5991
  ROC-AUC     : 0.6644
  Precision   : 0.1601
  Recall      : 0.6519
  F1-Score    : 0.2571
              precision    recall  f1-score   support

           0       0.93      0.59      0.73     83438
           1       0.16      0.65      0.26      9934

    accuracy                          

2026-07-26 15:52:02,870 | INFO | src.models.stage2_logistic | Saved evaluation metrics -> C:\Users\RABIYA BUSHRA\OneDrive\Attachments\Desktop\MajorProject\Implementation\BarrierLens_MP_G25_P48\outputs\stage2_results\logistic_evaluation_results.csv


In [5]:
# Top odds-ratio predictors per target
for target_col in STAGE2_TARGETS:
    if target_col not in results.get('targets', {}):
        print(f'Skipped {TARGET_DISPLAY.get(target_col, target_col)} (no analytic sample)')
        continue
    coefs = results['targets'][target_col]['coefficients']
    print(f'\n=== {TARGET_DISPLAY.get(target_col, target_col)} — top odds ratios ===')
    print(coefs.head(10).to_string(index=False))


=== Unmet Family Planning Need — top odds ratios ===
               Feature  Coefficient  OddsRatio
          v501_married     0.315614   1.371101
         v743f_missing     0.235262   1.265241
       v159_not at all     0.133069   1.142329
            v013_45-49     0.129843   1.138650
            v013_40-44     0.117484   1.124664
          v190_poorest     0.117055   1.124181
household_barrier_prob     0.102232   1.107641
      v717_not working     0.090095   1.094279
        v130_christian     0.082216   1.085690
            v013_25-29     0.077391   1.080465
